# Параметры сцены

In [7]:
import numpy as np
from particles import init_in_grid, one_step_simple

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

square_side = 50.
wall_width = 2.

bounds = np.array([[  0,   0],
                   [2 * square_side + wall_width,  square_side]])   # [x_min/y_min ; x_max/y_max]

wall = np.array([square_side, 0., wall_width, square_side])   # [x_min, y_min, width, height]

N = 60              # число частиц в КАЖДОЙ группе (итого 2N)
N = int(np.sqrt(N)) ** 2
d = 2.

# --- Константы сценария ---
N_STEPS = 1000
DIFFUSION_START = 340                 # стена начинает исчезать
WALL_SHRINK_PER_FRAME = 0.07           # на сколько уменьшаем высоту за кадр

# --- Инициализация частиц ---
rs1, vs1 = init_in_grid(bounds, N, random_vels=True, position="left",  density_factor=5)
rs2, vs2 = init_in_grid(bounds, N, random_vels=True, position="right", density_factor=5)

vs = np.vstack((vs1, vs2))
rs = np.vstack((rs1, rs2))    # итоговые позиции 2N
vs *= 0.66

# --- Оформление ---
DARKGRAY = "#222"
GRAY = "#333"
CIRC_COLOR_1 = "#7E348D"
CIRC_COLOR_2 = "#386c8e"
FIGWIDTH = 7

# Без полосы градиента

## Запуск симуляции (PyQt6)

In [12]:
import matplotlib.colors as mcolors  # <-- NEW

%matplotlib qt

ratio = bounds[1, 1] / bounds[1, 0]
fig, ax = plt.subplots()
fig.set_figwidth(FIGWIDTH)
fig.set_figheight(FIGWIDTH * ratio + 0.2)
fig.set_facecolor(DARKGRAY)

# ===== 20 px поля вокруг диаграммы (точно в пикселях) =====
M_px = 40
dpi = fig.get_dpi()
fw_px, fh_px = fig.get_size_inches() * dpi  # размеры фигуры в пикселях
left   = M_px / fw_px
bottom = M_px / fh_px
width  = 1.0 - 2*left
height = 1.0 - 2*bottom
ax.set_position([left, bottom, width, height])  # оси занимают центр с полями по 20 px


for n in range(N_STEPS):
    ax.clear()

    # --- DIFFUSION: стена усыхает ---
    if n >= DIFFUSION_START:
        wall[3] = max(0.0, wall[3] - WALL_SHRINK_PER_FRAME)
        wall[1] += WALL_SHRINK_PER_FRAME / 2

    # --- Рисуем стену ---
    ax.add_patch(Rectangle((wall[0], wall[1]), wall[2], wall[3],
                           facecolor=DARKGRAY, edgecolor='none', zorder=0))

    # --- Рисуем частицы ---
    for i, (x, y) in enumerate(rs):
        color = CIRC_COLOR_1 if i < N else CIRC_COLOR_2
        ax.add_patch(Circle((x, y), radius=d/2, facecolor=color,
                            edgecolor='none', zorder=1))

    # --- Шаг симуляции ---
    one_step_simple(rs, vs, d, bounds, [wall])

    # --- Оформление осей/поля ---
    ax.set_xlim(bounds[:, 0])
    ax.set_ylim(bounds[:, 1])
    ax.set_aspect('equal', adjustable='box')
    ax.set_facecolor(GRAY)
    ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.pause(0.05)


KeyboardInterrupt: 

## Запись в MP4

In [ ]:
from matplotlib.animation import FFMpegWriter
import matplotlib as mpl
import imageio_ffmpeg

FPS = 60
FRAMES = N_STEPS
FRAMES = 1300
PATH = "./outputs/diffusion-5.mp4"
DPI = 600
BITRATE = 6000

plt.ioff()  # пишем в файл, без интерактива
ratio = bounds[1, 1] / bounds[1, 0]
fig, ax = plt.subplots()
fig.set_figwidth(FIGWIDTH)
fig.set_figheight(FIGWIDTH * ratio + 0.2)
fig.set_facecolor(DARKGRAY)

# ===== 20 px поля вокруг диаграммы (точно в пикселях) =====
M_px = 40
dpi = fig.get_dpi()
fw_px, fh_px = fig.get_size_inches() * dpi  # размеры фигуры в пикселях
left   = M_px / fw_px
bottom = M_px / fh_px
width  = 1.0 - 2*left
height = 1.0 - 2*bottom
ax.set_position([left, bottom, width, height])  # оси занимают центр с полями по 20 px

mpl.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

writer = FFMpegWriter(
    fps=FPS,
    codec="libx264",
    bitrate=BITRATE,  # поднимайте при необходимости
    metadata={"title": "Diffusion", "artist": "ordevoir"},
    extra_args=[
                "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
                "-pix_fmt", "yuv420p", "-movflags", "faststart"]  # совместимость и быстрый старт
)

with writer.saving(fig, PATH, DPI):
    for n in range(FRAMES):
        ax.clear()

        # --- DIFFUSION: стена усыхает ---
        if n >= DIFFUSION_START:
            wall[3] = max(0.0, wall[3] - WALL_SHRINK_PER_FRAME)
            wall[1] += WALL_SHRINK_PER_FRAME / 2

        # --- Рисуем стену ---
        ax.add_patch(Rectangle((wall[0], wall[1]), wall[2], wall[3],
                            facecolor=DARKGRAY, edgecolor='none', zorder=0))

        # --- Рисуем частицы ---
        for i, (x, y) in enumerate(rs):
            color = CIRC_COLOR_1 if i < N else CIRC_COLOR_2
            ax.add_patch(Circle((x, y), radius=d/2, facecolor=color,
                                edgecolor='none', zorder=1))

        # --- Шаг симуляции ---
        one_step_simple(rs, vs, d, bounds, [wall])

        # --- Оформление осей/поля ---
        ax.set_xlim(bounds[:, 0])
        ax.set_ylim(bounds[:, 1])
        ax.set_aspect('equal', adjustable='box')
        ax.set_facecolor(GRAY)
        ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
        for spine in ax.spines.values():
            spine.set_visible(False)

        # --- Сохранение кадра ---
        fig.canvas.draw()
        writer.grab_frame()

print(f"Готово: {PATH}")

Готово: ./outputs/diffusion-5.mp4


## Тримминг

In [ ]:
from utils import trim_mp4

trim_mp4("./outputs/diffusion-5.mp4", 
         "./outputs/diffusion-5_trimmed.mp4", 1, 22, accurate=True)

Running: c:\Users\ordevoir\miniconda3\envs\marl\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe -y -i outputs\diffusion-5.mp4 -ss 00:00:01.000 -t 21.000 -map 0 -c:v libx264 -crf 18 -preset medium -c:a aac -b:a 192k -movflags +faststart outputs\diffusion-5_trimmed.mp4
Saved: outputs\diffusion-5_trimmed.mp4


## Компрессия

In [ ]:
from utils import compress_mp4

compress_mp4("./outputs/diffusion-5_trimmed.mp4", 
               "./outputs/diffusion-5_trimmed_compressed.mp4", scale=0.5)

Running:
 'c:\Users\ordevoir\miniconda3\envs\marl\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe' -hide_banner -loglevel error -y -i ./outputs/diffusion-5_trimmed.mp4 -c:v libx264 -preset slow -crf 18 -pix_fmt yuv420p -vf 'scale=trunc(iw*0.5/2)*2:trunc(ih*0.5/2)*2' -c:a copy -movflags faststart ./outputs/diffusion-5_trimmed_compressed.mp4


'./outputs/diffusion-5_trimmed_compressed.mp4'

# С полосой градиента

## Настройки полосы концентрации

In [14]:
import matplotlib.colors as mcolors

CELL_COUNT = 51           # 21 ячейка
CELL_W = wall_width       # ширина каждой ячейки = wall_width
STRIP_GAP = wall_width    # отступ под контейнером
STRIP_H = wall_width      # высота полосы
# базовые цвета двух типов частиц (те же, что у кругов)
COL_A = mcolors.to_rgb(CIRC_COLOR_1)          # тип A (первые N)
COL_B = mcolors.to_rgb(CIRC_COLOR_2)          # тип B (последние N)
COL_BG = mcolors.to_rgb(DARKGRAY)          # фон для «ослабления» по плотности
DENSITY_GAMMA = 1.0                        # <1 — усиливает слабые плотности, >1 — подавляет

def mix_colors(ca, cb, pa):
    ca, cb = np.asarray(ca), np.asarray(cb)
    return tuple(pa * ca + (1 - pa) * cb)

# чтобы центрировать полосу по ширине контейнера
W = bounds[1, 0] - bounds[0, 0]
strip_total_w = CELL_COUNT * CELL_W
strip_x0 = bounds[0, 0] + (W - strip_total_w) / 2.0  # левый край полосы
# вертикальное положение полосы (ниже контейнера)
STRIP_TOP = 0.0 - STRIP_GAP
STRIP_BOTTOM = STRIP_TOP - STRIP_H

def mix_colors(ca, cb, pa):
    """Линейное смешение цветов: pa — доля типа A (0..1)."""
    ca = np.asarray(ca)
    cb = np.asarray(cb)
    return tuple(pa * ca + (1 - pa) * cb)

# === История для усреднения полосы концентрации ===
HIST_LEN = 60  # окно усреднения в кадрах
hist_totals = np.zeros((CELL_COUNT, HIST_LEN), dtype=np.int32)  # сколько частиц в ячейке
hist_nA     = np.zeros((CELL_COUNT, HIST_LEN), dtype=np.int32)  # сколько из них типа A
hist_idx = 0        # куда писать в кольцевой буфер
frames_seen = 0     # сколько кадров уже накоплено (до HIST_LEN)


## Запуск симуляции (PyQt6)

In [5]:
%matplotlib qt

# --- Инициализация частиц ---
rs1, vs1 = init_in_grid(bounds, N, random_vels=True, position="top-left",  density_factor=8)
rs2, vs2 = init_in_grid(bounds, N, random_vels=True, position="bottom-right", density_factor=8)


vs = np.vstack((vs1, vs2))
rs = np.vstack((rs1, rs2))    # итоговые позиции 2N
vs *= 0.66

wall = np.array([square_side, 0., wall_width, square_side])   # [x_min, y_min, width, height]

ratio = bounds[1, 1] / bounds[1, 0]
fig, ax = plt.subplots()
fig.set_figwidth(FIGWIDTH)
fig.set_figheight(FIGWIDTH * ratio + 0.2)
fig.set_facecolor(DARKGRAY)

# ===== 20 px поля вокруг диаграммы (точно в пикселях) =====
M_px = 40
dpi = fig.get_dpi()
fw_px, fh_px = fig.get_size_inches() * dpi  # размеры фигуры в пикселях
left   = M_px / fw_px
bottom = M_px / fh_px
width  = 1.0 - 2*left
height = 1.0 - 2*bottom
ax.set_position([left, bottom, width, height])

for n in range(N_STEPS):
    ax.clear()

    # --- DIFFUSION: стена усыхает ---
    if n >= DIFFUSION_START:
        wall[3] = max(0.0, wall[3] - WALL_SHRINK_PER_FRAME)
        wall[1] += WALL_SHRINK_PER_FRAME / 2

    # --- Рисуем стену ---
    ax.add_patch(Rectangle((wall[0], wall[1]), wall[2], wall[3],
                           facecolor=DARKGRAY, edgecolor='none', zorder=0))

    # --- Рисуем частицы ---
    for i, (x, y) in enumerate(rs):
        color = CIRC_COLOR_1 if i < N else CIRC_COLOR_2
        ax.add_patch(Circle((x, y), radius=d/2, facecolor=color,
                            edgecolor='none', zorder=1))

    # ==== ПОЛОСА КОНЦЕНТРАЦИИ (усреднение по последним HIST_LEN кадрам) ====
    # фон под полосой
    ax.add_patch(Rectangle((bounds[0, 0], STRIP_TOP),
                           bounds[1, 0] - bounds[0, 0],
                           0.0 - STRIP_TOP,
                           facecolor=DARKGRAY, edgecolor='none', zorder=0.25))

    # --- 1) Подсчёт по текущему кадру (totals и nA для каждой ячейки) ---
    totals = np.zeros(CELL_COUNT, dtype=np.int32)
    nAs    = np.zeros(CELL_COUNT, dtype=np.int32)

    in_y = (rs[:, 1] >= 0.0) & (rs[:, 1] <= bounds[1, 1])  # вертикальный фильтр один раз
    for k in range(CELL_COUNT):
        x0 = strip_x0 + k * CELL_W
        x1 = x0 + CELL_W
        in_x = (rs[:, 0] >= x0) & (rs[:, 0] < x1)
        mask = in_x & in_y

        total = int(mask.sum())
        totals[k] = total
        if total > 0:
            nAs[k] = int(mask[:N].sum())   # первые N — тип A
        else:
            nAs[k] = 0

    # --- 2) Обновляем кольцевую историю и считаем средние ---
    hist_totals[:, hist_idx] = totals
    hist_nA[:,     hist_idx] = nAs
    hist_idx = (hist_idx + 1) % HIST_LEN
    frames_seen += 1
    m = min(frames_seen, HIST_LEN)  # фактическая длина окна (первые кадры < HIST_LEN)

    # Средняя плотность в ячейке (по total) за окно
    totals_avg = hist_totals[:, :m].mean(axis=1)

    # Средняя доля A за окно: суммируем nA и total, затем pA = sum(nA)/sum(total)
    nA_sum   = hist_nA[:,   :m].sum(axis=1).astype(float)
    tot_sum  = hist_totals[:, :m].sum(axis=1).astype(float)
    pA_avg   = np.divide(nA_sum, tot_sum, out=np.full(CELL_COUNT, 0.5), where=tot_sum > 0)

    # --- 3) Нормировка насыщенности по максимуму средних тоталов ---
    max_total_avg = float(totals_avg.max()) if totals_avg.size and totals_avg.max() > 0 else 1.0

    # --- 4) Рисуем ячейки по усреднённым значениям ---
    for k in range(CELL_COUNT):
        hue_rgb = np.asarray(mix_colors(COL_A, COL_B, pA_avg[k]))
        dens = (totals_avg[k] / max_total_avg) if max_total_avg > 0 else 0.0
        dens = dens ** DENSITY_GAMMA
        final_rgb = tuple(dens * hue_rgb + (1.0 - dens) * np.asarray(COL_BG))

        x0 = strip_x0 + k * CELL_W
        ax.add_patch(Rectangle((x0, STRIP_BOTTOM), CELL_W, STRIP_H,
                               facecolor=final_rgb, edgecolor='none', zorder=0.5))


    # --- Шаг симуляции ---
    one_step_simple(rs, vs, d, bounds, [wall])

    # --- Оформление осей/поля ---
    ax.set_xlim(bounds[:, 0])
    # важно: показать область ниже нуля, где полоса
    ax.set_ylim(STRIP_BOTTOM, bounds[1, 1])
    ax.set_aspect('equal', adjustable='box')
    ax.set_facecolor(GRAY)
    ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.pause(0.05)

: 

## Запись в MP4

In [15]:
from matplotlib.animation import FFMpegWriter
import matplotlib as mpl
import imageio_ffmpeg

# --- Инициализация частиц ---
rs1, vs1 = init_in_grid(bounds, N, random_vels=True, position="top-left",  density_factor=8)
rs2, vs2 = init_in_grid(bounds, N, random_vels=True, position="bottom-right", density_factor=8)

vs = np.vstack((vs1, vs2))
rs = np.vstack((rs1, rs2))    # итоговые позиции 2N
vs *= 0.66

wall = np.array([square_side, 0., wall_width, square_side])   # [x_min, y_min, width, height]

FPS = 60
FRAMES = 2000                      # используем это значение в цикле
PATH = "./outputs/diffusion-strip_4.mp4"
DPI = 600                        # вместо 600, чтобы не «висло»
BITRATE = 6000

plt.ioff()
ratio = bounds[1, 1] / bounds[1, 0]
fig, ax = plt.subplots()
fig.set_figwidth(FIGWIDTH)
fig.set_figheight(FIGWIDTH * ratio + 0.2)
fig.set_facecolor(DARKGRAY)

# поля в пикселях
M_px = 40
dpi = fig.get_dpi()
fw_px, fh_px = fig.get_size_inches() * dpi
left   = M_px / fw_px
bottom = M_px / fh_px
width  = 1.0 - 2*left
height = 1.0 - 2*bottom
ax.set_position([left, bottom, width, height])

mpl.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

writer = FFMpegWriter(
    fps=FPS,
    codec="libx264",
    bitrate=BITRATE,
    metadata={"title": "Diffusion", "artist": "ordevoir"},
    extra_args=["-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
                "-pix_fmt", "yuv420p", "-movflags", "faststart"]
)

with writer.saving(fig, PATH, DPI):
    for n in range(FRAMES):      # <-- важно: цикл по FRAMES
        ax.clear()

        # --- физика текущего кадра (исчезание стены) ---
        if n >= DIFFUSION_START:
            wall[3] = max(0.0, wall[3] - WALL_SHRINK_PER_FRAME)
            wall[1] += WALL_SHRINK_PER_FRAME / 2

        # --- рисуем стену ---
        ax.add_patch(Rectangle((wall[0], wall[1]), wall[2], wall[3],
                               facecolor=DARKGRAY, edgecolor='none', zorder=0))

        # --- рисуем частицы ---
        for i, (x, y) in enumerate(rs):
            color = CIRC_COLOR_1 if i < N else CIRC_COLOR_2
            ax.add_patch(Circle((x, y), radius=d/2, facecolor=color,
                                edgecolor='none', zorder=1))

    # ==== ПОЛОСА КОНЦЕНТРАЦИИ (усреднение по последним HIST_LEN кадрам) ====
        # фон под полосой
        ax.add_patch(Rectangle((bounds[0, 0], STRIP_TOP),
                            bounds[1, 0] - bounds[0, 0],
                            0.0 - STRIP_TOP,
                            facecolor=DARKGRAY, edgecolor='none', zorder=0.25))

        # --- 1) Подсчёт по текущему кадру (totals и nA для каждой ячейки) ---
        totals = np.zeros(CELL_COUNT, dtype=np.int32)
        nAs    = np.zeros(CELL_COUNT, dtype=np.int32)

        in_y = (rs[:, 1] >= 0.0) & (rs[:, 1] <= bounds[1, 1])  # вертикальный фильтр один раз
        for k in range(CELL_COUNT):
            x0 = strip_x0 + k * CELL_W
            x1 = x0 + CELL_W
            in_x = (rs[:, 0] >= x0) & (rs[:, 0] < x1)
            mask = in_x & in_y

            total = int(mask.sum())
            totals[k] = total
            if total > 0:
                nAs[k] = int(mask[:N].sum())   # первые N — тип A
            else:
                nAs[k] = 0

        # --- 2) Обновляем кольцевую историю и считаем средние ---
        hist_totals[:, hist_idx] = totals
        hist_nA[:,     hist_idx] = nAs
        hist_idx = (hist_idx + 1) % HIST_LEN
        frames_seen += 1
        m = min(frames_seen, HIST_LEN)  # фактическая длина окна (первые кадры < HIST_LEN)

        # Средняя плотность в ячейке (по total) за окно
        totals_avg = hist_totals[:, :m].mean(axis=1)

        # Средняя доля A за окно: суммируем nA и total, затем pA = sum(nA)/sum(total)
        nA_sum   = hist_nA[:,   :m].sum(axis=1).astype(float)
        tot_sum  = hist_totals[:, :m].sum(axis=1).astype(float)
        pA_avg   = np.divide(nA_sum, tot_sum, out=np.full(CELL_COUNT, 0.5), where=tot_sum > 0)

        # --- 3) Нормировка насыщенности по максимуму средних тоталов ---
        max_total_avg = float(totals_avg.max()) if totals_avg.size and totals_avg.max() > 0 else 1.0

        # --- 4) Рисуем ячейки по усреднённым значениям ---
        for k in range(CELL_COUNT):
            hue_rgb = np.asarray(mix_colors(COL_A, COL_B, pA_avg[k]))
            dens = (totals_avg[k] / max_total_avg) if max_total_avg > 0 else 0.0
            dens = dens ** DENSITY_GAMMA
            final_rgb = tuple(dens * hue_rgb + (1.0 - dens) * np.asarray(COL_BG))

            x0 = strip_x0 + k * CELL_W
            ax.add_patch(Rectangle((x0, STRIP_BOTTOM), CELL_W, STRIP_H,
                                facecolor=final_rgb, edgecolor='none', zorder=0.5))

        # --- оформление осей КАЖДЫЙ кадр ---
        ax.set_xlim(bounds[:, 0])
        ax.set_ylim(STRIP_BOTTOM, bounds[1, 1])
        ax.set_aspect('equal', adjustable='box')
        ax.set_facecolor(GRAY)
        ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
        for spine in ax.spines.values():
            spine.set_visible(False)

        # --- сохранить кадр ---
        writer.grab_frame()

        # --- шаг симуляции ПОСЛЕ сохранения текущего кадра ---
        one_step_simple(rs, vs, d, bounds, [wall])

print(f"Готово: {PATH}")

Готово: ./outputs/diffusion-strip_4.mp4


In [16]:
import winsound
winsound.PlaySound('WindowsUnlock', winsound.SND_ALIAS)

In [ ]:
# from utils import trim_mp4

# trim_mp4("./outputs/diffusion-strip_2.mp4", 
#          "./outputs/diffusion-strip_2_trimmed.mp4", 1, 25, accurate=True)

Running: c:\Users\ordevoir\miniconda3\envs\marl\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe -y -i outputs\diffusion-strip_2.mp4 -ss 00:00:01.000 -t 24.000 -map 0 -c:v libx264 -crf 18 -preset medium -c:a aac -b:a 192k -movflags +faststart outputs\diffusion-strip_2_trimmed.mp4
Saved: outputs\diffusion-strip_2_trimmed.mp4


In [17]:
from utils import crop_mp4

crop_mp4("./outputs/diffusion-strip_4.mp4", "./outputs/diffusion-strip_4_cropped.mp4", 
         (200, 200, 0, 0))

In [18]:
from utils import compress_mp4

compress_mp4("./outputs/diffusion-strip_4_cropped.mp4", 
               "./outputs/diffusion-strip_4_cropped_compressed.mp4", scale=0.5)

Running:
 'c:\Users\ordevoir\miniconda3\envs\marl\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe' -hide_banner -loglevel error -y -i ./outputs/diffusion-strip_4_cropped.mp4 -c:v libx264 -preset slow -crf 18 -pix_fmt yuv420p -vf 'scale=trunc(iw*0.5/2)*2:trunc(ih*0.5/2)*2' -c:a copy -movflags faststart ./outputs/diffusion-strip_4_cropped_compressed.mp4


'./outputs/diffusion-strip_4_cropped_compressed.mp4'